In [1]:
from meetAgent import meetWorkFlow
from calender_agent import callWorkFlow
from mailAgent import gmailWorkFlow
from codingAgent import codeAgent
from dotenv import load_dotenv 
from langchain_community.tools import DuckDuckGoSearchRun
load_dotenv()

c:\Users\Prasanna\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from deepagents import create_deep_agent, CompiledSubAgent
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-20b")
internet_search = DuckDuckGoSearchRun()

meet_subagent = CompiledSubAgent(
    name="meet-analyzer",
    description="Specialized agent for meet transcript summary and query response",
    runnable=meetWorkFlow
)

calender_subagent = CompiledSubAgent(
   name="calander-event-handler",
   description="specilized agent for adding events and task on the google calender",
   runnable= callWorkFlow
)

mails_subagent = CompiledSubAgent(
    name="gmail_handler",
    description="agent specilized for drafting, sending, summurizing received mails",
    runnable=gmailWorkFlow
)

code_subagent = CompiledSubAgent(
    name="code_executor",
    description="Executes Python programs securely in sandbox.",
    runnable=codeAgent,
)

In [ ]:
subagents = [code_subagent,mails_subagent,calender_subagent,meet_subagent]
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.sqlite import SqliteSaver
from deepagents.middleware.memory import MemoryMiddleware
from langchain.agents.middleware import SummarizationMiddleware
import sqlite3

 
conn = sqlite3.connect("deepagent_state.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)
store = InMemoryStore()

composite_backend = lambda rt: CompositeBackend(
    default=StateBackend(rt),
    routes={
        "/memories/": StoreBackend(rt),
    }
)
 
agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="you are main-agent having multiple subagents with calender, gmail, python code exucator and google meet trancscropts retriver deleget control to them as needed",
    subagents=subagents,
    backend=composite_backend,
    store=store,
    checkpointer=checkpointer,
    middleware=[ SummarizationMiddleware(
            model=model,
            trigger=("tokens", 4000),
            keep=("messages", 20),
        ),
    ],
   
)

In [21]:
config = {"configurable":{"thread_id":"11grhwj6qdt65"}}
result = agent.invoke({"messages":
    [{"role": "user",
    "content":"draft a mail on the topic of your name is found in epstine files and in the content just say i am just joking and send this mail sumitkasale2005@gmail.com"}]
},config=config)

In [22]:
result

{'messages': [HumanMessage(content='draft a mail on the topic of your name is found in epstine files and in the content just say i am just joking and send this mail sumitkasale2005@gmail.com', additional_kwargs={}, response_metadata={}, id='8bec6afb-75d1-4c69-bdf3-3962a1f9340b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants a draft email. They want to send to sumitkasale2005@gmail.com. They want content: "Your name is found in Epstein files" and then "I am just joking". They want to send it. They want us to use gmail_handler subagent? The user wants to send email. The instructions: "draft a mail on the topic of your name is found in epstine files and in the content just say i am just joking and send this mail sumitkasale2005@gmail.com". So we need to send an email. We have a subagent type gmail_handler. We can call task to launch a gmail_handler to send email. Provide details: subject: "Your name is found in Epstein files" or something. Body: "I am just j